<a href="https://colab.research.google.com/github/shivashankarb2006/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shivashankarb2006/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [7]:
!git clone https://github.com/shivashankarb2006/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 236, done.
remote: Counting objects: 100% (236/236), done.
remote: Compressing objects: 100% (180/180), done.
remote: Total 236 (delta 126), reused 113 (delta 40), pack-reused 0 (from 0)
Receiving objects: 100% (236/236), 2.00 MiB | 7.17 MiB/s, done.
Resolving deltas: 100% (126/126), done.


In [9]:
!ls flyrank-ml-internship/data/raw

content_refresh_anonymized.csv


In [10]:
import pandas as pd

df = pd.read_csv(
    "flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"
)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [11]:
feature_cols = [
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "search_volume",
    "competition",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

feature_cols = [col for col in feature_cols if col in df.columns]

X = df[feature_cols].copy()

for col in X.columns:
    if pd.api.types.is_numeric_dtype(X[col]):
        X[col] = X[col].fillna(X[col].median())

print("Features used:")
print(feature_cols)

print("\nFeature matrix shape:", X.shape)
print("\nMissing values:")
print(X.isna().sum())

Features used:
['impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'search_volume', 'competition', 'word_count', 'char_count', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']

Feature matrix shape: (30000, 14)

Missing values:
impressions_prev_30d      0
clicks_prev_30d           0
sessions_prev_30d         0
search_volume             0
competition               0
word_count                0
char_count                0
content_age_days          0
days_since_last_update    0
ctr                       0
avg_position              0
engagement_rate           0
scroll_rate               0
ai_traffic_pct            0
dtype: int64


In [12]:
future_or_leaky_cols = [
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "trend_direction",
    "trend_pct"
]

print("Leakage / future-information check")
print("-----------------------------------")

for col in future_or_leaky_cols:
    if col in df.columns:
        print(f"EXCLUDED: {col}")
    else:
        print(f"Not present: {col}")

print("\nFinal feature columns:")
print(feature_cols)

Leakage / future-information check
-----------------------------------
EXCLUDED: impressions_last_30d
EXCLUDED: clicks_last_30d
EXCLUDED: sessions_last_30d
EXCLUDED: trend_direction
EXCLUDED: trend_pct

Final feature columns:
['impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'search_volume', 'competition', 'word_count', 'char_count', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']


### Feature notes

- `impressions_prev_30d`: impressions during the previous 30-day period. Available before the outcome period.
- `clicks_prev_30d`: clicks during the previous 30-day period. Available before the outcome period.
- `sessions_prev_30d`: sessions during the previous 30-day period. Available before the outcome period.
- `search_volume`: observed search demand for the content topic.
- `competition`: observed competition level.
- `word_count`: content length.
- `char_count`: character count of the content.
- `content_age_days`: age of the content page.
- `days_since_last_update`: time since the last update.
- `ctr`: historical click-through rate.
- `avg_position`: historical average search position.
- `engagement_rate`: historical engagement signal.
- `scroll_rate`: historical scrolling signal.
- `ai_traffic_pct`: historical AI-traffic percentage.

Missing numeric values are filled using the median of the available values. The selected features are intended to represent information available before the outcome period.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [13]:
# Columns that describe the outcome period or are derived from future movement
future_or_leaky_cols = [
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "trend_direction",
    "trend_pct"
]

print("Leakage / future-information check")
print("-----------------------------------")

for col in future_or_leaky_cols:
    if col in df.columns:
        print(f"EXCLUDED: {col}")
    else:
        print(f"Not present: {col}")

print("\nFinal feature columns:")
print(feature_cols)

Leakage / future-information check
-----------------------------------
EXCLUDED: impressions_last_30d
EXCLUDED: clicks_last_30d
EXCLUDED: sessions_last_30d
EXCLUDED: trend_direction
EXCLUDED: trend_pct

Final feature columns:
['impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'search_volume', 'competition', 'word_count', 'char_count', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']


### Excluded fields

- `impressions_last_30d` — excluded because it represents the outcome period and can directly leak the target.
- `clicks_last_30d` — excluded because it belongs to the outcome period.
- `sessions_last_30d` — excluded because it belongs to the outcome period.
- `trend_direction` — excluded because it summarizes performance movement and may contain information from the outcome period.
- `trend_pct` — excluded because it directly describes performance change and can leak the outcome.

The goal is to use only information that would be available before making the prioritization decision.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.